Massive parallel programming on GPUs and applications, by Lokman ABBAS TURKI  

# 1. Device query and error handling

## 1.1 Objective

The main purpose of this lab is to introduce the student to the CUDA hardware resources and their capabilities. The specification of the GPU will be frequently needed in the following labs. Thus, once the file DevQuery.cu has been configured with the appropriate instructions, it can be compiled and executed whenever students require information about the limitations of the currently utilized device.

This lab session serves as the initial opportunity for students to utilize CUDA documentation, enabling them to discover:
1) the specifications of CUDA API functions within the [CUDA_Runtime_API](https://docs.nvidia.com/cuda/cuda-runtime-api/index.html).
2) the examples of how to use the CUDA API functions in [CUDA_C_Programming_Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html)


## 1.2 Content

Compile DevQuery.cu using

In [ ]:
!nvcc DevQuery.cu -o DQ

Execute DQ using (on Microsoft Windows OS ./ is not needed)

In [2]:
!./DQ

As long as you did not include any additional instruction in the file DevQuery.cu, the execution above is not supposed to return any value. At least, no compilation error is detected by the compiler!

In the following questions you will need to include your own code in the file DevQuery.cu, then compile it and execute it before answering.


### 1.2.1 In the main function, use cudaGetDeviceCount to display the number of available GPUs

Hint: use the CUDA documentation


### 1.2.2 In the main function, and with device 0, use cudaGetDeviceProperties to display

a) The global memory size. What is the largest single-precision floating-point array that can be allocated on the GPU RAM?

**Answer**: $15637086208$ B. A single-precision floating points is $32$ bits long, or $4$ bytes. Therefore, you must divide the global memory size by $4$ to know how many floating-point you can store in the RAM.

b) The maximum number maxGridSize of blocks that can be launched. What do you notice?

**Answer**: $2147483647 \times 65535 \times 65535$. All three limits can be achieved simultaneously, contrarily to `maxThreadsDim`, see just below. I don't know what to notice. https://forums.developer.nvidia.com/t/newbie-question-about-maximum-number-of-blocks/42015

c) The maximum numbers `maxThreadsDim` and `maxThreadsPerBlock` of threads per block that can be launched. How do you understand the values given?

**Answer**: `maxThreadsDim` is an array containing the maximum number you can allocate per dimensions of the block. `maxThreadsPerBlock` is the actual number of threads you can allocate on the block at run time. Since `maxThreadsPerBlock` $\leq$ `maxThreadsDim[0]` x `maxThreadsDim[1]` x `maxThreadsDim[2]`, you can never use all the space virtually available. It is up to the operator to dispatch threads on the three coordinates.

d) The size of a warp in terms of threads.

**Answer**: $32$, as usual. This is due to the design of the hardware.

e) The shared memory size per block.

**Answer:** This is the amount of memory shared by threads in a block. In Bytes.

f) The number of registers per block.

**Answer:** $65536$. The number of $32-$ bit registers per block. Note that $65536/128=512$.

g) The size of 2D textures. Would it be possible to have it entirely in the cache?

**Answer:** No, there is way too much bytes for the RAM to handle. Remember that GPU RAM saturates faster than CPU RAM.

h) The number of CUDA cores. (hint: each multiprocessor contains usually 128 CUDA cores)

**Answer:** You have to multiply the number of multiprocessor by $128$.


### 1.2.3 Analyze the results from Section 1.2.2

a) How long should be the sequences of diverged execution of warps? (hint: search for "diverge" keyword in CUDA documentation)

**Answer**: What is CUDA Thread Divergence? Thread divergence, also called "warp divergence" or "branch divergence," is a computation bottleneck that occurs when some subset of threads in a warp take a different path at a control flow branch, such as an if statement or loop conditional test. It should be as short as possible.


b) How do you justify that the number of threads should be (and not must be) a power of 2?

**Answer**: It is not a question of structural design, but of *performance* considerations. Parallelization algorithm often requires halving works recursively. Consider, for example, a binary tree. It is naturally balanced when the number of leaves is a power of two, and many divide-and-conquer algorithms are based on this data structure. However, I would like to emphasise that, in practice, the number of threads does not need to be a power of two.

c) What can limit a program from launching the maximum number of blocks?

**Answer**:This is largely constrained by the capacity of the SM themselves. A SM has a maximum capacity on the number of blocks it can managed.

d) What can limit a program from launching the maximum number of threads?

**Answer**: The shared memory capacity in a block, the maximum number of threads per SM, the maximum number of registers.

e) What can trivially make us find a compromise between the number of blocks and the number of threads to be used?

**Answer**: This is the GPU occupancy. Occupancy is defined as the ratio of the number of active warps to the maximum number of warps that can be active on a given streaming multiprocessor (SM) at any one time. 

*   A large number of threads per block means more warps and parallelism, but less registers per thread.
*   A large number of blocks means shared memory of SM is split further. Also, if all blocks are too large, the SM might not able to reach its full theoretical occupancy. For example, given the information displayed above, since a SM can run only 1024 threads at a time, it can contain only one single full block.

### 1.2.4 Recall cudaGetDeviceProperties on the device 20

a) What if you change the $0$ argument in `cudaGetDeviceProperties` by 20?

**Answer**: The output message makes no sense.

b) What kind of `cudaError_t` do you get?

**Answer**: I receive no error message.

c) Use now testCUDA and see what happens.

**Answer**: I get the error message: *The number of devices available is 1 GPUS. There is an error in file DevQuery.cu at line 30*.